In [1]:
%%bash
pip install statsmodels
pip install patsy
# pip install dask 
# pip install dask distributed --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 133.7 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 110.5 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 117.6 MB/s eta 0:00:00


In [15]:
import os
import glob
import pandas as pd
import sys
import statsmodels.api as sm
import numpy as np
from scipy import stats
from scipy.stats import zscore
from mwas_func import get_pheno_df

# from dask.distributed import Client
# import dask.dataframe as dd
# import glob
# import multiprocessing


In [4]:
# python3 mwas_03.1_assoc.py $batch $PHE_MAP $ICD $COV $OUT

'-f'

In [4]:
## passed arguments
batch   = 50 # sys.argv[1] # 1,2,3 .. 50 -> from for loop in .sh 
phe_map = 'phecode_map.csv' # sys.argv[2] # phecode mapping file
icdfile = 'icd.phe' # sys.argv[3] # cases file
covfile = 'cov.phe' # sys.argv[4] # covariates/controls file
outsfx  = '_batch50.csv' # sys.argv[5] # output df filename suffix


In [30]:
%%bash

WGTLIST_FULL=EBB.BRAIN.METHYL.HERIT/EBB.BRAIN.METHYL.HERIT.list # list of paths to weights (hsq < 0.05)

mkdir -p TMP
split -n l/50 --numeric-suffixes=1 $WGTLIST_FULL TMP/temp_WGTLIST_ # split weights list into chunks

for f in TMP/temp_WGTLIST_0*; do
  mv $f $(echo $f | sed 's/_0/_/')
done



In [43]:
%%bash

batch=13

## get same CpGs per batch as in the brain-based analysis
WGTLIST=TMP/temp_WGTLIST_$batch # brain-based CpG list

#ed 's|.*/\(.*\)\.wgt\.RDat|\1|' $WGTLIST > matched_list.$batch
codes=$(sed 's|.*/\(.*\)\.wgt\.RDat|\1|' $WGTLIST)

# Convert the extracted codes into a pattern for grep
pattern=$(echo "$codes" | tr '\n' '|')
pattern="${pattern%|}"  # Remove the trailing pipe

# Use grep to extract matching rows from file2
grep -E "$pattern" EPICDB.BLOOD.METHYL.HERIT/scores/scores.list > matched_list.$batch

wc -l $WGTLIST
wc -l matched_list.$batch

2119 TMP/temp_WGTLIST_13
657 matched_list.13


In [44]:
657 / 2119

0.31005191127890513

In [7]:
## load predictions files - for all phenotypes
print('loading brain-based predictions ...')
brain_predfile = 'batch' + str(batch) + '_predict.txt'
brain_pred = pd.read_csv(brain_predfile, sep="\t").rename(columns={'IID':'eid'}).drop(columns=['FID']).set_index('eid')


loading brain-based predictions ...


In [6]:
print('loading blood-based predictions ...')
blood_predfile = 'batch' + str(batch) + '_predict_blood.txt'
blood_pred = pd.read_csv(blood_predfile, sep="\t").rename(columns={'IID':'eid'}).drop(columns=['FID']).set_index('eid')


loading blood-based predictions ...


In [8]:
#set(brain_pred.columns.values, blood_pred.columns.values)
common_cpgs = list(blood_pred.columns.intersection(brain_pred.columns))
n_common_cpgs = len(common_cpgs)
print(f'{n_common_cpgs} CpG sites both in brain- & blood-based DNAm predictors')

607 CpG sites both in brain- & blood-based DNAm predictors


In [13]:
## load phecodes mapping file
phecodes = pd.read_csv(phe_map)
phecodes = phecodes[['phecode', 'description']].drop_duplicates()
n_pheno  = phecodes.shape[0] 

## set covariates
PCS = [f"PC{i}" for i in range(1, 21)]
covariates = ['sex', 'age', 'batch'] + PCS


In [16]:
## start phenotype j analysis
j=0
pheno_name_j  = phecodes.iloc[j]['phecode']
description_j = phecodes.iloc[j]['description']

## load phenotype j file
print(f'starting analysis for {description_j} ({pheno_name_j}) | {j} / {n_pheno}:')
print(f'\tloading phenotype/covariates')

phe_j = get_pheno_df(
    pheno=pheno_name_j,
    icdfile=icdfile,
    covfile=covfile,
    covariates = covariates
)

starting analysis for Intestinal infection (GI_526) | 0 / 445:
	loading phenotype/covariates


In [17]:
## match the order of pred matrix eids
phe_j = phe_j.reindex(brain_pred.index)
blood_pred = blood_pred.reindex(brain_pred.index)

In [18]:
Y_orig  = phe_j[pheno_name_j]
X_covar = sm.add_constant(phe_j[covariates])
Y_resid = sm.Logit(Y_orig, X_covar).fit(disp=0).resid_response

## get n cases /controls
n_cas = len(Y_orig[Y_orig==1])
n_cnt = len(Y_orig[Y_orig==0])

## Association analysis
cpgs   = list(blood_pred.columns.intersection(brain_pred.columns))
n_cpgs = len(cpgs)
print(f"\t{description_j} ~ CpGs associations; {n_cpgs} CpGs")

res = []

y = Y_resid.values


	Intestinal infection ~ CpGs associations; 607 CpGs


In [19]:
for i in range(n_cpgs):
        
    ###  get scaled CpG i data
    cpg_i = cpgs[i]
    
    # blood-based predictor
    x1 = zscore(blood_pred[cpg_i], nan_policy='omit')
    # x1 = blood_pred[cpg_i]
    
    # brain-based predictor 
    x2 = zscore(brain_pred[cpg_i], nan_policy='omit')
    # x2 = brain_pred[cpg_i]
    
    # mask missing
    mask = ~np.isnan(x1) & ~np.isnan(x2) & ~np.isnan(y)
    x1_, x2_, y_ = x1[mask], x2[mask], y[mask]
    n = len(y_)

    if n < 3:
        continue

    # design matrix
    X = np.column_stack((x1_, x2_))
    X = sm.add_constant(X)

    model = sm.OLS(y_, X).fit()

    # coefficients
    beta1, beta2 = model.params[1], model.params[2]
    se1, se2     = model.bse[1], model.bse[2]
    pval1, pval2 = model.pvalues[1], model.pvalues[2]

    ci = model.conf_int()

    # partial correlations 
    sd_y  = np.std(y_, ddof=1)
    sd_x1 = np.std(x1_, ddof=1)
    sd_x2 = np.std(x2_, ddof=1)

    r_x1 = beta1 * sd_x1 / sd_y
    r_x2 = beta2 * sd_x2 / sd_y
    
    # brain-blood predictor correlation 
    r_x12 = stats.pearsonr(x1_, x2_)[0]

    res.append({
        "cpg": cpg_i,
        
        # x1
        "beta_x1": beta1,
        "se_x1": se1,
        "lci_x1": ci[1, 0],
        "uci_x1": ci[1, 1],
        "zscore_x1": beta1 / se1,
        "pvalue_x1": pval1,
        "r_x1": r_x1,

        # x2
        "beta_x2": beta2,
        "se_x2": se2,
        "lci_x2": ci[2, 0],
        "uci_x2": ci[2, 1],
        "zscore_x2": beta2 / se2,
        "pvalue_x2": pval2,
        "r_x2": r_x2,

        "n_cas": n_cas,
        "n_cnt": n_cnt,
        
        "r_x12": r_x12
        
    })
    
res_df = pd.DataFrame(res)
res_df

KeyboardInterrupt: 

In [20]:
res_df = pd.DataFrame(res)
res_df

,cpg,beta_x1,se_x1,lci_x1,uci_x1,zscore_x1,pvalue_x1,r_x1,beta_x2,se_x2,lci_x2,uci_x2,zscore_x2,pvalue_x2,r_x2,n_cas,n_cnt,r_x12
0,cg07399572,0.000845,0.001031,-0.001176,0.002865,0.819302,0.412614,0.003746,-0.000357,0.001031,-0.002378,0.001663,-0.346743,0.728784,-0.001586,23691,383915,0.939503
1,cg25115154,0.000337,0.000386,-0.000418,0.001093,0.874895,0.381631,0.001497,-0.000253,0.000386,-0.001009,0.000502,-0.657270,0.511008,-0.001124,23691,383915,0.402094
2,cg27079975,-0.000051,0.000357,-0.000750,0.000648,-0.142271,0.886866,-0.000225,-0.000252,0.000357,-0.000951,0.000447,-0.705500,0.480500,-0.001116,23691,383915,-0.140191
3,cg06816651,-0.000315,0.000357,-0.001016,0.000385,-0.882749,0.377373,-0.001399,0.000310,0.000357,-0.000391,0.001010,0.866595,0.386164,0.001374,23691,383915,0.154432
4,cg07435449,0.000467,0.000566,-0.000642,0.001576,0.825362,0.409166,0.002072,-0.000340,0.000566,-0.001449,0.000770,-0.600258,0.548335,-0.001507,23691,383915,0.781565
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148,cg01797501,0.000276,0.000597,-0.000894,0.001445,0.461780,0.644240,0.001223,0.000095,0.000597,-0.001074,0.001265,0.159579,0.873213,0.000422,23691,383915,0.806209
149,cg20280170,0.000268,0.000423,-0.000562,0.001098,0.633173,0.526621,0.001189,0.000067,0.000423,-0.000763,0.000897,0.158721,0.873889,0.000298,23691,383915,0.551967
150,cg19499581,-0.000206,0.000449,-0.001087,0.000675,-0.458741,0.646420,-0.000914,0.000657,0.000449,-0.000224,0.001538,1.462388,0.143636,0.002915,23691,383915,0.618579
151,cg01106572,0.000611,0.000661,-0.000685,0.001907,0.923459,0.355768,0.002709,0.000102,0.000661,-0.001194,0.001399,0.154895,0.876904,0.000454,23691,383915,0.845526


In [23]:
a = pd.read_csv('GI_526__batch50.csv')

In [22]:
brain_pred[cpg_i]

eid
1000015    0.00
1000027    0.00
1000039    0.00
1000040    0.00
1000053    0.00
           ... 
6024817    0.00
6024825   -2.27
6024832    0.00
6024848    0.00
6024854    0.00
Name: cg00358278, Length: 407606, dtype: float64